In [ ]:
import os
import scanpy as sc
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import random
from LingerGRN.pseudo_bulk import *
from LingerGRN.preprocess import *
import LingerGRN.LINGER_tr as LINGER_tr
import LingerGRN.LL_net as LL_net
from LingerGRN.TF_activity import *

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 5), facecolor='white')
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

root = path_to_wd
os.chdir(root)

## Prepare the input data

In [ ]:
adata_full = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

In [ ]:
lineage_list = ["Global", 'Malignant', 'CD4_T', 'CD8_T_NK_ILC', 'B', 'Myeloid', 'MSC']
lineage = 'Malignant' ## Choose one from lineage_list

if lineage == "Global":
    adata_use = adata_full.copy()
elif lineage == "Malignant":
    adata_use = adata_full[(adata_full.obs["lineage"] == "Epithelial") & (~adata_full.obs["celltype"].isin(["Normal"]))].copy()
else:
    adata_use = adata_full[adata_full.obs["lineage"] == lineage].copy()

In [ ]:
dir_path = f'{root}Reproducibility/Results/LINGER/Primary/{lineage}/'
os.makedirs(dir_path, exist_ok = True)

outdir=f"{dir_path}output/"
os.makedirs(outdir, exist_ok = True)

datadir=f"{dir_path}data/"
os.makedirs(datadir, exist_ok = True)

post_BCG = ['BC_027','BC_032',"BC_039","BC_040",'BC_043',"BC_044","BC_047","BC_048"]
adata = adata_use[~adata_use.obs['sample'].isin(post_BCG)].copy()

adata.obs["label"] = adata.obs["celltype"]
adata.obs['barcode'] = adata.obs_names

label = pd.DataFrame({'barcode_use': adata.obs_names,
                      'label': adata.obs["label"]
                     })

In [ ]:
adata_RNA  = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_ATAC = adata[:, adata.var.modality == "Peaks"].copy()

adata_RNA = adata_RNA[:, adata_RNA.var['black_list']=='keep']
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

adata_RNA.var['gene_ids'] = adata_RNA.var_names
adata_ATAC.var['gene_ids'] = adata_ATAC.var_names

adata_RNA.raw = adata_RNA
adata_ATAC.raw = adata_ATAC

## Generate the pseudo-bulk/metacell 

In [ ]:
value_counts = adata_RNA.obs["sample"].value_counts()

cell_number_thresh = {
    "Global": 100, 
    'Malignant': 100,
    'CD4_T': 0,
    'CD8_T_NK_ILC': 0, 
    'B': 15, 
    'Myeloid': 0,
    'MSC': 0
}

samples_to_keep = value_counts[value_counts > cell_number_thresh[lineage]].index

# Filter the AnnData object to keep only these samples
adata_RNA_filt = adata_RNA[adata_RNA.obs["sample"].isin(samples_to_keep)]
adata_ATAC_filt = adata_ATAC[adata_ATAC.obs["sample"].isin(samples_to_keep)]

In [ ]:
def pseudo_bulk_modified(adata_RNA,adata_ATAC,singlepseudobulk):
    K = 20
    adata_RNA=adata_RNA_filt[adata_RNA_filt.obs['sample']==tempsample].copy()
    adata_ATAC=adata_ATAC_filt[adata_ATAC_filt.obs['sample']==tempsample].copy()
    sc.pp.normalize_total(adata_RNA, target_sum=1e4)
    sc.pp.log1p(adata_RNA)
    adata_RNA.raw=adata_RNA 
    sc.pp.highly_variable_genes(adata_RNA, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable]
    sc.pp.scale(adata_RNA, max_value=10)
    sc.tl.pca(adata_RNA, n_comps=15,svd_solver="arpack")
    pca_RNA=adata_RNA.obsm['X_pca']
    sc.pp.log1p(adata_ATAC)
    adata_ATAC.raw=adata_ATAC
    sc.pp.highly_variable_genes(adata_ATAC, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_ATAC = adata_ATAC[:, adata_ATAC.var.highly_variable]
    sc.pp.scale(adata_ATAC, max_value=10, zero_center=True)
    sc.tl.pca(adata_ATAC, n_comps=15,svd_solver="arpack")
    pca_ATAC=adata_ATAC.obsm['X_pca']
    pca = np.concatenate((pca_RNA,pca_ATAC), axis=1)
    adata_RNA.obsm['pca']=pca
    adata_ATAC.obsm['pca']=pca
    sc.pp.neighbors(adata_RNA, n_neighbors=K, n_pcs=30,use_rep='pca')
    connectivities=(adata_RNA.obsp['distances']>0)
    import random
    label=pd.DataFrame(adata_RNA.obs['label'])
    label.columns=['label']
    label.index=adata_RNA.obs['barcode'].tolist()
    #label=label['label'].values
    cluster=list(set(label['label'].values))
    allindex=[]
    # np.random.seed(42)  # Set seed for reproducibility
    random.seed(42)  # Set seed for reproducibility
    for i in range(len(cluster)):
        temp=label[label['label']==cluster[i]].index
        N = len(temp) # Total number of elements
        if N>=10:
            m = int(np.floor(np.sqrt(N)))+1  # Number of elements to sample
            if singlepseudobulk>0:
                m=2 
            sampled_elements = random.sample(range(N), m)
            temp=temp[sampled_elements]
            allindex=allindex+temp.tolist()
    connectivities=pd.DataFrame(connectivities.toarray(),index=adata_RNA.obs['barcode'].tolist())
    connectivities=connectivities.loc[allindex].values
    A=(connectivities @ adata_RNA.raw.X.toarray())
    TG_filter1=A/(K-1)
    RE_filter1=(connectivities @ adata_ATAC.raw.X.toarray())/(K-1)
    TG_filter1=pd.DataFrame(TG_filter1.T,columns=allindex,index=adata_RNA.raw.var['gene_ids'].tolist())
    RE_filter1=pd.DataFrame(RE_filter1.T,columns=allindex,index=adata_ATAC.raw.var['gene_ids'].tolist())
    return TG_filter1,RE_filter1


In [ ]:
samplelist=list(set(adata_ATAC_filt.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]
TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])
singlepseudobulk = (adata_RNA_filt.obs['sample'].unique().shape[0]*adata_RNA_filt.obs['sample'].unique().shape[0]>100)
for tempsample in samplelist:
    adata_RNAtemp=adata_RNA_filt[adata_RNA_filt.obs['sample']==tempsample]
    adata_ATACtemp=adata_ATAC_filt[adata_ATAC_filt.obs['sample']==tempsample]
    TG_pseudobulk_temp,RE_pseudobulk_temp=pseudo_bulk_modified(adata_RNAtemp,adata_ATACtemp,singlepseudobulk)                
    TG_pseudobulk=pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk=pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    RE_pseudobulk[RE_pseudobulk > 100] = 100

In [ ]:
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)

#TG_pseudobulk.to_csv(f"{datadir}TG_pseudobulk.tsv")
#RE_pseudobulk.to_csv(f"{datadir}RE_pseudobulk.tsv")

adata_ATAC_filt.var['gene_ids'].to_csv(f"{datadir}Peaks.txt",header=None,index=None)

## Overlap the region with general GRN

In [ ]:
os.chdir(dir_path)

genome='hg38'
method = 'LINGER'
Datadir=os.path.expanduser('~/reference/LINGER/') 
GRNdir=os.path.expanduser('~/reference/LINGER/data_bulk/')

preprocess(TG_pseudobulk,RE_pseudobulk,GRNdir,genome,method,outdir)

## Train for the LINGER model

In [ ]:
activef='ReLU'
species = 'Human'

LINGER_tr.training(GRNdir,method,outdir,activef,species)   

## Cell population gene regulatory network

In [ ]:
genome = 'hg38'
network = 'cell population'

# TF binding potential
LL_net.TF_RE_binding(GRNdir,adata_RNA_filt,adata_ATAC_filt,genome,method,outdir)
# cis-regulatory network
LL_net.cis_reg(GRNdir,adata_RNA_filt,adata_ATAC_filt,genome,method,outdir)
# trans-regulatory network
LL_net.trans_reg(GRNdir,method,outdir,genome)

TF_activity = regulon(outdir,adata_RNA,GRNdir,network,genome)
TF_activity = TF_activity.dropna(how='all').dropna(how='all', axis=1)
TF_activity.to_csv(f"Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_{lineage}.txt", sep='\t')

In [ ]:
TF_use = np.intersect1d(adata_RNA.var_names, TF_activity.index)

# RNA
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)
sc.pp.scale(adata_RNA, max_value=10)

# TF_activity
adata_TF = sc.AnnData(TF_activity.T.loc[adata_RNA.obs_names,:].copy(), obs=adata_RNA.obs)
sc.pp.scale(adata_TF, max_value=10)

exp_df = pd.DataFrame(adata_RNA[:, TF_use].X, index=adata_RNA.obs_names, columns=TF_use)
TF_df  = pd.DataFrame(adata_TF[:, TF_use].X,  index=adata_TF.obs_names, columns=TF_use)

TF_df.to_csv(f"Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_zscore_{lineage}.txt",sep='\t')

In [ ]:
for tmp_celltype in adata_RNA.obs['label'].unique():
    CB_use = adata_RNA[adata_RNA.obs['label'].isin([tmp_celltype])].obs_names
    exp_df_tmp = exp_df.loc[CB_use,:].copy().T
    TF_df_tmp = TF_df.loc[CB_use,:].copy().T
    ## average
    exp_TF_df = pd.DataFrame(index = exp_df_tmp.index, columns = ['EXP', 'TF'])
    exp_TF_df.loc[:,'EXP'] = exp_df_tmp.mean(axis='columns')
    exp_TF_df.loc[:,'TF'] = TF_df_tmp.mean(axis='columns')
    exp_TF_df.to_csv(f"Reproducibility/Results/LINGER/Primary/cell_population_exp_TF_activity_zscore_{tmp_celltype}.txt",sep='\t')

## TF activity inference from TCGA-BLCA bulk RNA-seq

In [ ]:
Datadir=os.path.expanduser('~/reference/LINGER/') 
GRNdir=os.path.expanduser('~/reference/LINGER/data_bulk/')
genome='hg38'
dir_path = f'Reproducibility/Results/LINGER/Primary/Global/'
os.chdir(dir_path)
outdir=dir_path+'output/'
network = 'general'

In [ ]:
adata_TCGA = sc.read("Reproducibility/Data/External_cohort/TCGA_BLCA_RBAseq_FPKM_for_LINGER.h5ad")
TF_activity = regulon(outdir,adata_TCGA,GRNdir,network,genome)
TF_activity = TF_activity.dropna(how='all').dropna(how='all', axis=1)
TF_activity.to_csv(f"Reproducibility/Results/LINGER/Primary/TCGA_BLCA_TF_activity_general.txt",sep='\t')